In [ ]:
import decoupler as dc
import warnings
warnings.filterwarnings("ignore")

import decoupler as dc 
import pandas as pd
import scanpy as sc
import numpy as np
from pyslingshot import Slingshot
import matplotlib.pyplot as plt
from py_monocle import (
    learn_graph,
    order_cells,
    compute_cell_states,
    regression_analysis,
    differential_expression_genes,
)
import scanpy as sc
import methods as mt

In [ ]:
adata = sc.read_h5ad("/home/roger/data_itx/adata_all.h5ad")
umap_cd4 = pd.read_csv("/home/roger/Baixades/tots_cd4_umap_coordinates.csv")
umap_cd8 = pd.read_csv("/home/roger/Baixades/totscd8_umap_coordinates.csv")

In [ ]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad("/home/roger/data_itx/adata_all.h5ad")

umap_cd4 = pd.read_csv("/home/roger/Baixades/tots_cd4_umap_coordinates.csv")
umap_cd8 = pd.read_csv("/home/roger/Baixades/totscd8_umap_coordinates.csv")


def subset_with_umap(adata, umap_df):
    umap_df = umap_df.set_index("cell")[["UMAP_1", "UMAP_2"]]

    common = adata.obs_names.intersection(umap_df.index)

    adata_sub = adata[common].copy()
    adata_sub.obsm["X_umap"] = umap_df.loc[adata_sub.obs_names].values

    print(f"Kept {adata_sub.n_obs} cells")

    return adata_sub


adata_cd4 = subset_with_umap(adata, umap_cd4)
adata_cd8 = subset_with_umap(adata, umap_cd8)

In [ ]:
# Subset data based on interests, as we know that the trajectories differe between conditions, we are going to change that.
cd8_clusters = ["CD8_ex", "CD8_mem", "CD8_eff", "CD8_naive", "CD8_ex_act", "CD8_act"]
adata = adata[adata.obs['celltype'].isin(cd8_clusters)]
#adata = adata[adata.obs['treatment'] == "post"]

In [ ]:
umap_cd4 = pd.read_csv("/home/roger/Baixades/tots_cd4_umap_coordinates.csv")
umap_cd8 = pd.read_csv("/home/roger/Baixades/totscd8_umap_coordinates.csv")

In [ ]:
import numpy as np
import pandas as pd

# Make sure cell column is unique
if umap_cd8["cell"].duplicated().any():
    raise ValueError("There are duplicated cell names in umap_cd8['cell'].")

# Use cell names as index
umap_df = umap_cd8.set_index("cell")

# Check overlap with adata cells
missing_in_umap = adata.obs_names.difference(umap_df.index)
extra_in_umap = umap_df.index.difference(adata.obs_names)

print(f"Cells in adata: {adata.n_obs}")
print(f"Cells in umap_cd8: {umap_df.shape[0]}")
print(f"Missing cells in UMAP table: {len(missing_in_umap)}")
print(f"Extra cells in UMAP table: {len(extra_in_umap)}")

In [ ]:
import numpy as np
import pandas as pd

# Set cell names as index
umap_df = umap_cd8.set_index("cell")

# Keep only cells present in both adata and umap_cd8
common_cells = adata.obs_names.intersection(umap_df.index)

print(f"Original adata cells: {adata.n_obs}")
print(f"UMAP table cells: {umap_df.shape[0]}")
print(f"Common cells kept: {len(common_cells)}")

# Filter adata to common cells
adata = adata[common_cells].copy()

# Add UMAP coordinates, ordered exactly like adata.obs_names
adata.obsm["X_umap"] = umap_df.loc[
    adata.obs_names,
    ["UMAP_1", "UMAP_2"]
].to_numpy()

# Check
print(adata)
print(adata.obsm["X_umap"].shape)

In [ ]:
sc.pl.umap(adata, color=["celltype"])

sc.tl.pca(adata, n_comps=16)
sc.pp.neighbors(adata, n_pcs=16)
sc.tl.leiden(adata, resolution=0.7)
sc.tl.umap(adata, min_dist=0.3) 


In [ ]:
# try to assess the batch correction using harmony or scvi? and then save the object always 

In [ ]:
sc.pl.umap(adata, color=["celltype"])

In [ ]:
# we should define a palette for our analysis.
palette_cd4 = {
        "Naive":  "#4C72B0",   # blue
        "Tfh":    "#DD8452",   # orange
        "Th17":   "#55A868",   # green
        "Treg":   "#C44E52",   # red
    }
    
palette_cd8 = {
        "CD8_naive": "skyblue",
        "CD8_mem": "green",
        "CD8_ex": "red",
        "CD8_ex_act": "purple",
        "CD8_eff": "navy",
        "CD8_act": "gold",
    }


In [ ]:
mt.monocle3(adata, palette_cd8)

In [ ]:
mt.run_paga(adata, root_celltype="CD8_naive")

In [ ]:
# work with claude to make the different figures and see what i can do... 

In [ ]:
start_node=4
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
custom_xlim = (-12, 12)
custom_ylim = (-12, 12)
# plt.setp(axes, xlim=custom_xlim, ylim=custom_ylim)

slingshot = Slingshot(adata, celltype_key="celltype", obsm_key="X_umap", start_node=start_node, is_debugging="verbose")

slingshot.fit(num_epochs=1, debug_axes=axes)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 4))
axes[0].set_title("Clusters")
axes[1].set_title("Pseudotime")
slingshot.plotter.curves(axes[0], slingshot.curves)
slingshot.plotter.clusters(axes[0], labels=np.arange(slingshot.num_clusters), s=4, alpha=0.5)
slingshot.plotter.clusters(axes[1], color_mode="pseudotime", s=5)

In [ ]:
def run_slingshot(adata, start_node):
    slingshot = Slingshot(adata, celltype_key="celltype", obsm_key="X_umap", start_node=start_node, is_debugging="verbose")

    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
    custom_xlim = (-12, 12)
    custom_ylim = (-12, 12)

    fig, axes = plt.subplots(ncols=2, figsize=(12, 4))
    axes[0].set_title("Clusters")
    axes[1].set_title("Pseudotime")
    slingshot.plotter.curves(axes[0], slingshot.curves)
    slingshot.plotter.clusters(axes[0], labels=np.arange(slingshot.num_clusters), s=4, alpha=0.5)
    slingshot.plotter.clusters(axes[1], color_mode="pseudotime", s=5)
    return fig

